In [1]:
import pandas as pd
import math
from ReadNBS import read_file_complet2,WriteNBS2
from IPython.display import Image
import numpy as np

In [2]:
file_in = "in/abys.nbs"
file_out = 'in/abys3.nbs'


class MusicData:
    def __init__(self):
        #self.read_file(file_in)

        self.file_loaded = False
        

    def read_file(self, file_in):
        self.header, self.data, self.fin = read_file_complet2(file_in)
        self.file_loaded = True
        
        self.directory = os.path.dirname(file_in)
        self.file_name = os.path.splitext(os.path.basename(file_in))[0]
        
        self.process_initial_data()
        self.adjust_layers()
        #self.modify_instrument_data()
        
        return self.file_name
        
        
        
    def get_tempo(self):
        return self.header['tempo']/100
    
    def set_tempo(self, tempo):
        self.header['tempo'] = int(tempo*100)
        
    def get_tempos(self):
        closest_delay = int(np.floor(1/(self.get_tempo()/4)*100//5)*5)
        delays = [closest_delay - 5, closest_delay, closest_delay + 5]
        self.tempos = [100/(d)*4 for d in delays]
        text = []
        for i in range(3):
            if(delays[i]%10 == 0):
                text.append(f"{self.tempos[i]:.2f} : Exact bpm")
            else:
                text.append(f"{self.tempos[i]:.2f} : Swing bpm")
        return text
        
    def upadte_tempo(self, index):
        self.speed_up(self.tempos[index])
        self.set_tempo(20)
    
    def speed_up(self,tick_second):
        self.new_data['real tick'] = [math.ceil(a*20/tick_second) for a in self.new_data['tick']]

    def process_initial_data(self):
        self.data['octave'] = self.data['key'] // 12
        self.data['note'] = self.data['key'] % 12
        self.data['real tick'] = self.data['tick']

    def adjust_layers(self):
        tick = 0
        layer = 0
        for i in range(self.data.shape[0]):
            if self.data.iloc[i]['tick'] > tick:
                tick = self.data.iloc[i]['tick']
                layer = 0
            if self.data.iloc[i]['layer'] > layer:
                self.data.at[i, 'layer'] = layer
            layer += 1

    def modify_instrument_data(self, modifier):
        nbs_instruments = {'didgeridoo': 12, 'bass': 1, 'guitar': 5, 'banjo': 14, 'pling': 15, 'iron_xylophone': 10,
                       'bit': 13, 'harp': 0, 'cow_bell': 11, 'flute': 6, 'chime': 8, 'xylophone': 9, 'bell': 7}

        octave_instruments = {'didgeridoo': -2, 'bass': -2, 'guitar': -1, 'banjo': 0, 'pling': 0, 'iron_xylophone': 0,
                       'bit': 0, 'harp': 0, 'cow_bell': 1, 'flute': 1, 'chime': 2, 'xylophone': 2, 'bell': 2}

        new_rows = []
        tick = 0
        layer = 0
        for i in range(self.data.shape[0]):
            if self.data.iloc[i]['tick'] > tick:
                tick = self.data.iloc[i]['tick']
                layer = 0

            for instr_i in range(13):
                note = self.data.iloc[i].copy()
                octave = max(-3, min(note['octave'], 4))

                if modifier[octave+3, instr_i]:
                    note['insts'] = nbs_instruments[list(octave_instruments.keys())[instr_i]]
                    instr_octave = octave_instruments[list(octave_instruments.keys())[instr_i]]
                    if instr_octave < octave:
                        note['key'] = note['note'] + 12
                    elif instr_octave + 1 == octave:
                        note['key'] = note['note'] + 12
                    else:
                        note['key'] = note['note']

                    note['layer'] = layer
                    layer += 1
                    new_rows.append(note)

        self.new_data = pd.DataFrame(new_rows)
        self.final_layer_adjustment()
    
    def final_layer_adjustment(self):
        tick = 0
        layer = 0
        for i in range(self.new_data.shape[0]):
            if self.new_data.iloc[i]['tick'] > tick:
                tick = self.new_data.iloc[i]['tick']
                layer = 0
            if self.new_data.iloc[i]['layer'] > layer:
                self.new_data.at[i, 'layer'] = layer
            layer += 1

    def write_nbs(self):
        if(self.file_loaded):
            full_file_name  = self.directory+'/'+self.file_name+".nbs"
            print("Saving " + full_file_name)
            WriteNBS2(self.new_data,full_file_name, self.header, self.fin)
            print("file saved")



In [3]:
import sys
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QGridLayout, QLabel, QPushButton,
                             QFileDialog, QVBoxLayout,QHBoxLayout, QCheckBox,QLineEdit,QComboBox)
from PyQt5.QtCore import Qt
import os


class CustomCheckBox(QCheckBox):
    def __init__(self, octave, value, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.octave = octave
        self.value = value
        self.setStyleSheet(self.get_stylesheet())

    def get_stylesheet(self):
        cross_style = ("QCheckBox::indicator:checked::before {"
                       "content: '✕';"
                       "color: white;"
                       "font-weight: bold;"
                       "position: absolute;"
                       "top: -2px;"
                       "left: 3px;"
                       "}")

        if self.octave < self.value:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: black; }"
                    "QCheckBox::indicator:checked { background-color: gray; }"
                    + cross_style)
        elif self.octave == self.value or self.octave == self.value + 1:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: blue; }"
                    "QCheckBox::indicator:checked { background-color: green; }"
                    + cross_style)
        else:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: yellow; }"
                    "QCheckBox::indicator:checked { background-color: orange; }"
                    + cross_style)

class CheckboxGridWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        
        self.processor = MusicData()
        
        self.setWindowTitle("Noteblock song program")
        
        # Central widget
        central_widget = QWidget(self)
        self.setCentralWidget(central_widget)
        
        # Vertical layout to hold the entire UI
        v_layout = QVBoxLayout()
        central_widget.setLayout(v_layout)

        # Button to load a file
        self.load_file_btn = QPushButton("Load NBS File")
        self.load_file_btn.clicked.connect(self.load_file)
        v_layout.addWidget(self.load_file_btn)

        # Label to display file name
        
        H_loaded_layout = QHBoxLayout()
        H_loaded_layout.addWidget(QLabel("Input File Name: "))
        self.file_label = QLabel("No file loaded")
        H_loaded_layout.addWidget(self.file_label)
        v_layout.addLayout(H_loaded_layout)
        
        # QEdit for save file name
        H_save_layout = QHBoxLayout()
        H_save_layout.addWidget(QLabel("Output File Name: "))
        self.output_file = QLineEdit("")
        H_save_layout.addWidget(self.output_file)
        v_layout.addLayout(H_save_layout)
        
        self.output_file.textChanged.connect(self.updateSaveFileName)
        
        
        #For the tempo
        
        self.input_tempo = QLabel("Input Tempo")
        v_layout.addWidget(self.input_tempo)
        
        self.fix_tempo = QCheckBox("Adjust Tempo")
        v_layout.addWidget(self.fix_tempo)
        
    
        self.choose_tempo = QComboBox()
        H_save_layout = QHBoxLayout()
        H_save_layout.addWidget(QLabel("Chose tempo:"))
        H_save_layout.addWidget(self.choose_tempo)
        v_layout.addLayout(H_save_layout)
        
        
        # Create a grid layout for checkboxes
        grid_layout = QGridLayout()
        v_layout.addLayout(grid_layout)

        # Dictionary of instruments with their values
        instruments = {'didgeridoo': -2, 'bass': -2, 'guitar': -1, 'banjo': 0, 'pling': 0, 'iron_xylophone': 0,
                       'bit': 0, 'harp': 0, 'cow_bell': 1, 'flute': 1, 'chime': 2, 'xylophone': 2, 'bell': 2}

        # Create the header row
        grid_layout.addWidget(QLabel('Octaves'), 0, 0, Qt.AlignCenter)
        for col, instrument in enumerate(instruments.keys(), start=1):
            label = QLabel(instrument)
            grid_layout.addWidget(label, 0, col)
            
            
        # Add the rows with checkboxes
        self.checkboxes = []
        for row,octave in enumerate([-3,-2,-1,0,1,2,3,4],start = 1):
            label = QLabel(str(octave))
            grid_layout.addWidget(label, row, 0, Qt.AlignCenter)
            row_checkboxes = []
            for col, (instrument, value) in enumerate(instruments.items(), start=1):
                checkbox = CustomCheckBox(octave, value)
                grid_layout.addWidget(checkbox, row, col, Qt.AlignCenter)
                row_checkboxes.append(checkbox)
            self.checkboxes.append(row_checkboxes)

        # Save button
        self.save_btn = QPushButton("Save")
        self.save_btn.clicked.connect(self.save_data)
        v_layout.addWidget(self.save_btn)

    def load_file(self):
        options = QFileDialog.Options()
        file_name, _ = QFileDialog.getOpenFileName(self, "Open File", "", "NBS (*.nbs)", options=options)
        
        if file_name:
            name = self.processor.read_file(file_name)
            name += "_updated"
            self.file_label.setText(name)
            self.output_file.setText(name)
            self.processor.file_name = name
            
            tempo = self.processor.header['tempo']/100
            self.input_tempo.setText(f"Imput tempo : {tempo:.2f}")
            self.update_tempo_box(self.processor.get_tempos())

    def save_data(self):
        # Initialize an array to hold the state of each checkbox
        checkbox_states = []

        # Loop through each row of checkboxes
        for row_checkboxes in self.checkboxes:
            row_states = []
            # Loop through each checkbox in the row
            for checkbox in row_checkboxes:
                # Append the state of the checkbox (True if checked, False if not)
                row_states.append(checkbox.isChecked())
            # Append the states of the current row to the main array
            checkbox_states.append(row_states)

        # Optionally, print the array to console to verify (you can remove this later)
        all_states = []
        for row_state in checkbox_states:
            all_states.append(row_state)

        # Here you can add additional code to save `checkbox_states` to a file or process it further
        self.processor.modify_instrument_data(np.array(all_states))
        
        
        # tempo change
        
        self.processor.upadte_tempo(self.choose_tempo.currentIndex())
        
        self.processor.write_nbs()

        
        
    def updateSaveFileName(self):
        # Récupérer le texte du QLineEdit
        text = self.output_file.text()
        self.processor.file_name = text

        
    def update_tempo_box(self, items, default_index=1): 
        
        self.choose_tempo.clear()  # Clear existing items
        self.choose_tempo.addItems(items)  # Add new items
        if len(items) > default_index:  # Ensure the default index is within the range
            self.choose_tempo.setCurrentIndex(default_index)
        

In [4]:
app = QApplication(sys.argv)
window = CheckboxGridWindow()
window.show()
sys.exit(app.exec_())

Loading file:  C:/Users/marce/Downloads/marry on a cross.nbs
Saving C:/Users/marce/Downloads/marry on a cross_updated.nbs
file saved
Saving C:/Users/marce/Downloads/marry on a cross_updated.nbs
file saved
Saving C:/Users/marce/Downloads/marry on a cross_updated.nbs
file saved
Saving C:/Users/marce/Downloads/marry on a cross_updated.nbs
file saved
Saving C:/Users/marce/Downloads/marry on a cross_updated.nbs
file saved


SystemExit: 0

D:\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
window.processor.data

In [ ]:
window.processor.new_data

In [ ]:
delays

In [ ]:
closest_delay = int(np.floor(1/(6.67/4)*100//5)*5)
delays = [closest_delay - 5, closest_delay, closest_delay + 5]
tempos = [100/(d)*4 for d in delays]
text = []
for i in range(3):
    if(delays[i]%10 == 0):
        print(f"{tempos[i]:.2f} : Exact bpm")
    else:
        print(f"{tempos[i]:.2f} : Swing bpm")